# RG3: 외부 이상탐지 방법을 동일 구간검증에 적용
Sources: johnwslee Mahalanobis/VAE; smartfactory-xai diverse detector agreement. Corrected implementations, not score reproduction.

In [1]:
from pathlib import Path
import json, numpy as np, pandas as pd, torch
from torch import nn
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.covariance import LedoitWolf
from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import confusion_matrix, average_precision_score, precision_recall_curve
torch.set_num_threads(2)
root=Path.cwd()
if not (root/'data').exists(): root=root.parent
t=root/'reports/tables'
d=pd.read_csv(t/'original_labeled_shot_features.csv',parse_dates=['TimeStamp'])
d=d[d['product'].eq('RG3')].reset_index(drop=True)
assert len(d)==591 and d.TimeStamp.is_unique
g=d.comparison_block.to_numpy(); blocks=np.unique(g)
cols=['Injection_Time','Filling_Time','Plasticizing_Time','Cycle_Time','Clamp_Close_Time','Cushion_Position','Plasticizing_Position','Max_Injection_Speed','Max_Screw_RPM','Max_Injection_Pressure','Max_Switch_Over_Pressure','Max_Back_Pressure','Average_Back_Pressure']+['Barrel_Temperature_'+str(k) for k in range(1,7)]+['Hopper_Temperature','Mold_Temperature_3','Mold_Temperature_4']
a=d[cols].replace([np.inf,-np.inf],np.nan).to_numpy(float)
class VAE(nn.Module):
 def __init__(self,p):
  super().__init__(); self.enc=nn.Sequential(nn.Linear(p,15),nn.ReLU(),nn.Linear(15,5),nn.ReLU());self.mu=nn.Linear(5,2);self.logvar=nn.Linear(5,2);self.dec=nn.Sequential(nn.Linear(2,5),nn.ReLU(),nn.Linear(5,15),nn.ReLU(),nn.Linear(15,p))
 def forward(self,x,sample=True):
  h=self.enc(x);mu=self.mu(h);lv=self.logvar(h).clamp(-10,5);z=mu+torch.randn_like(mu)*torch.exp(.5*lv) if sample else mu
  return self.dec(z),mu,lv
memo={}; audits=[]
def fit_score(excluded):
 key=tuple(sorted(excluded))
 if key in memo:return memo[key]
 normal=np.flatnonzero(d.PassOrFail.eq(0).to_numpy() & ~np.isin(g,key))
 imp=SimpleImputer(strategy='median',keep_empty_features=True).fit(a[normal]);x=imp.transform(a)
 keep=x[normal].std(axis=0)>1e-7
 sc=StandardScaler().fit(x[normal][:,keep]);x=np.clip(sc.transform(x[:,keep]),-20,20)
 models=[LedoitWolf().fit(x[normal]),IsolationForest(n_estimators=150,random_state=42,n_jobs=2).fit(x[normal]),OneClassSVM(nu=.01,gamma='scale').fit(x[normal]),LocalOutlierFactor(n_neighbors=20,novelty=True).fit(x[normal])]
 raw=[models[0].mahalanobis(x),-models[1].score_samples(x),-models[2].score_samples(x),-models[3].score_samples(x)]
 torch.manual_seed(42);m=VAE(x.shape[1]);opt=torch.optim.Adam(m.parameters(),lr=.001,weight_decay=.0001);xt=torch.tensor(x[normal],dtype=torch.float32)
 m.train()
 for epoch in range(80):
  for idx in torch.randperm(len(xt)).split(64):
   xb=xt[idx];re,mu,lv=m(xb);loss=((re-xb)**2).mean()+.01*(-.5*(1+lv-mu.square()-lv.exp()).sum(1)).mean()
   opt.zero_grad();loss.backward();opt.step()
 m.eval()
 with torch.no_grad():
  allx=torch.tensor(x,dtype=torch.float32);re,_,_=m(allx,sample=False);raw.append(((re-allx)**2).mean(1).numpy())
 # All rank references come only from reference normal rows, never heldout rows.
 rank=np.column_stack([np.searchsorted(np.sort(s[normal]),s,side='right')/(len(normal)+1) for s in raw])
 out={'mahalanobis':rank[:,0],'vae':rank[:,4],'diverse_mean':rank[:,[1,2,3,4]].mean(1)}
 votes=(rank[:,[1,2,3,4]]>=.99).sum(1)
 for k in range(1,5):out['consensus_'+str(k)]=(votes>=k).astype(float)
 out=pd.DataFrame(out)
 audits.append(dict(excluded_blocks='|'.join(key),normal_train=len(normal),feature_count=int(keep.sum()),overlap_rows=int(np.isin(g[normal],key).sum()),used_defect_train=0))
 memo[key]=out
 return out
def cut_f1(y,s):
 p,r,c=precision_recall_curve(y,s);f=2*p[:-1]*r[:-1]/np.maximum(p[:-1]+r[:-1],1e-12);k=max(range(len(c)),key=lambda k:(f[k],r[k],c[k]));return float(c[k]),float(f[k])
records=[];choices=[]
for task,reason,expected in [('gas','가스',17),('shortshot','미성형',8)]:
 mask=d.PassOrFail.eq(0)|d.Reason.str.contains(reason,na=False); ids=np.flatnonzero(mask)
 y=d.Reason.str.contains(reason,na=False).to_numpy(int);assert y[ids].sum()==expected
 for block in blocks:
  q=ids[g[ids]==block]; ref=ids[g[ids]!=block];outer=fit_score([block]).iloc[q]
  inner=pd.DataFrame(index=ref,columns=outer.columns,dtype=float)
  for ib in np.unique(g[ref]):
   qi=ref[g[ref]==ib];inner.loc[qi]=fit_score([block,ib]).iloc[qi].to_numpy()
  candidates=[]
  for name in outer.columns:
   if name.startswith('consensus'):cut=.5;pred=inner[name].to_numpy()>=cut;tp=((y[ref]==1)&pred).sum();fp=((y[ref]==0)&pred).sum();fn=((y[ref]==1)&~pred).sum();fi=2*tp/max(2*tp+fp+fn,1)
   else:cut,fi=cut_f1(y[ref],inner[name].to_numpy())
   candidates.append((fi,name,cut))
   for i,s in zip(q,outer[name]):records.append(dict(task=task,model=name,TimeStamp=d.TimeStamp.iloc[i],block=block,actual=int(y[i]),score=float(s),threshold=cut,prediction=int(s>=cut)))
  fi,name,cut=max(candidates,key=lambda z:(z[0],z[1]))
  choices.append(dict(task=task,block=block,model=name,inner_F1=fi,threshold=cut))
  for i,s in zip(q,outer[name]):records.append(dict(task=task,model='primary_inner_selected',TimeStamp=d.TimeStamp.iloc[i],block=block,actual=int(y[i]),score=float(s),threshold=cut,prediction=int(s>=cut)))
  print(task,block,'selected',name,flush=True)
p=pd.DataFrame(records);rows=[]
for (task,name),z in p.groupby(['task','model']):
 assert z.TimeStamp.is_unique and len(z)==(583 if task=='gas' else 574)
 tn,fp,fn,tp=confusion_matrix(z.actual,z.prediction,labels=[0,1]).ravel()
 rows.append(dict(task=task,model=name,TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/max(2*tp+fp+fn,1),AP=average_precision_score(z.actual,z.score)))
metrics=pd.DataFrame(rows)
p.to_csv(t/'rg3_external_methods_predictions.csv',index=False,encoding='utf-8-sig')
metrics.to_csv(t/'rg3_external_methods_metrics.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(choices).to_csv(t/'rg3_external_methods_inner_choices.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(audits).to_csv(t/'rg3_external_methods_training_audit.csv',index=False,encoding='utf-8-sig')
p[p.actual.eq(1)].to_csv(t/'rg3_external_methods_all_defects.csv',index=False,encoding='utf-8-sig')
print(metrics.to_string(index=False),flush=True)
report='# RG3 외부 방법 적용 — 정상 학습 및 중첩 구간검증\n\n'
report+='기존 591 Shot에서 가스는 정상566+가스17, 미성형은 정상566+미성형8로 각각 평가. 다른 불량은 해당 평가에서 제외. RH/LH 중복 Shot을 다시 늘리지 않았다.\n\n'
report+='5개 observation block 외부검증과 4개 내부구간검증. 연속 설비 run 확정값이 아닌 기존 관측 구간이다. 모든 정규화·공분산·신경망·점수 기준은 학습 정상만 사용. 임계값/모델 선택은 내부 OOF만 사용. 외부 불량 학습 및 Timestamp 중복 0. 이미 반복 열람한 자료라 새 데이터 독립검증은 아니다.\n\n'
report+='22개 원본 센서 후보에서 학습 정상 상수만 제거. Average_Screw_RPM은 기존 단위 이슈로 제외. 센서 입력은 위 노트북 cols에 명시.\n\n'
report+='Mahalanobis는 원본 역행렬 대신 LedoitWolf 축소 공분산. VAE는 22→15→5→mu/logvar 2→5→15→22, MSE+0.01 KL,80epoch,정상만 학습. 평가는 eval 및 잠재 평균으로 결정론적 복원. IF150트리,OCSVM nu.01,LOF20이웃. 합의는 4개(IF/SVM/LOF/VAE) 학습 정상99% 기준 통과 수 1~4; 원본 AE 합의를 수정 VAE로 대체한 비교이며 원본 성적의 재현은 아니다.\n\n'
report+='출처: https://github.com/johnwslee/injection_molding_analysis ; https://github.com/tkddnjs-dlqslek/smartfactory-xai . 원본 코드의 평가 라벨 임계값 선정, KL 누락 등을 그대로 복사하지 않았다. 비라벨 데이터는 이번에는 사용하지 않았다.\n\n'
report+=metrics.to_csv(index=False)
report+='\nprimary_inner_selected만 사전에 정의한 내부 선택 파이프라인 결과. 개별 후보 중 외부 성적 최고를 재선택하면 탐색 결과다. 기존 가스8/6도 탐색 후보여서 독립 우위 주장 불가.\n'
(root/'reports/rg3_external_methods_comparison.md').write_text(report,encoding='utf-8')


gas 20201021_001626_RG3 selected diverse_mean


gas 20201022_002405_RG3 selected diverse_mean


gas 20201022_044026_RG3 selected diverse_mean


gas 20201023_010128_RG3 selected mahalanobis


gas 20201023_044148_RG3 selected diverse_mean


...

shortshot 20201022_044026_RG3 selected vae


shortshot 20201023_010128_RG3 selected diverse_mean


shortshot 20201023_044148_RG3 selected consensus_2


     task                  model  TP  FP  FN  TN       F1       AP
      gas            consensus_1   8 368   9 198 0.040712 0.025450
      gas            consensus_2   0  34  17 532 0.000000 0.029160
      gas            consensus_3   0  11  17 555 0.000000 0.029160
      gas            consensus_4   0   4  17 562 0.000000 0.029160
      gas           diverse_mean   3  92  14 474 0.053571 0.031190
      gas            mahalanobis   7 260  10 306 0.049296 0.027506
      gas primary_inner_selected   3  92  14 474 0.053571 0.029594
      gas                    vae   5 203  12 363 0.044444 0.026226
shortshot            consensus_1   4 368   4 198 0.021053 0.012345
shortshot            consensus_2   0  34   8 532 0.000000 0.013937
shortshot            consensus_3   0  11   8 555 0.000000 0.013937
shortshot            consensus_4   0   4   8 562 0.000000 0.013937
shortshot           diverse_mean   4 345   4 221 0.022409 0.011115
shortshot            mahalanobis   3 304   5 262 0.019048 0.01

2026